# Pix3D Single-Category (Chair) Ablation Study
Notebook này được cấu hình để chạy trên Google Colab với GPU (khuyến nghị T4).

## Mục tiêu
1. Giải nén dataset Pix3D từ Google Drive và tiền xử lý riêng category `chair`.
2. Huấn luyện 4 cấu hình để làm rõ Ablation Study (chứng minh tác động của từng cải tiến).
3. Đánh giá và so sánh định tính (Visual Comparison) giữa Ground Truth và Predict.

## 1. Setup Environment & Mount Drive

In [ ]:
from google.colab import drive
import os

# Clone mã nguồn từ branch resnet50+mlp trên GitHub
!git clone -b resnet50+mlp https://github.com/TangDien02/AI_3D_Reconstruction_Systerm.git
os.chdir('AI_3D_Reconstruction_Systerm/project')
print("Current working directory:", os.getcwd())

# Cài đặt các thư viện cần thiết
!pip install -r requirements.txt
!apt-get install unrar

# Mount Google Drive để lấy file nén dataset pix3d.rar
drive.mount('/content/drive')

## 2. Chuẩn bị Dữ liệu (Extract & Preprocess)
Giả sử file dataset được đặt tại `/content/drive/MyDrive/Datasets/pix3d.rar`

In [ ]:
# Giải nén dữ liệu
!mkdir -p data/raw/pix3d
!unrar x -o+ /content/drive/MyDrive/Datasets/pix3d.rar data/raw/

# Tiền xử lý riêng category 'chair' để huấn luyện
!python src/preprocessing/build_processed_dataset.py \
    --raw-dir data/raw/pix3d \
    --output-dir data/processed_chair \
    --categories chair \
    --num-points 2048 \
    --overwrite

## 3. Huấn luyện (Ablation Study)
Chạy lần lượt các cấu hình để đánh giá sự thay đổi. Quá trình này sẽ sinh ra các model checkpoint trong folder `results/`.

In [ ]:
# Cấu hình 1: Baseline (ResNet50 + MLP cơ bản + Chamfer Loss thuần)
!python src/training/training_pipeline.py \
    --dataset-mode processed \
    --processed-dir data/processed_chair \
    --split train \
    --val-split val \
    --categories chair \
    --num-points 2048 \
    --epochs 80 \
    --batch-size 32 \
    --encoder-name resnet50 \
    --feature-dim 2048 \
    --decoder-type mlp \
    --freeze-encoder \
    --output-dir results/ablation_baseline \
    --early-stopping-patience 10

In [ ]:
# Cấu hình 2: + PEFT Adapter & Unfreeze (Giai đoạn 2 của chiến lược huấn luyện)
!python src/training/training_pipeline.py \
    --dataset-mode processed \
    --processed-dir data/processed_chair \
    --split train \
    --val-split val \
    --categories chair \
    --num-points 2048 \
    --epochs 80 \
    --batch-size 32 \
    --encoder-name resnet50 \
    --feature-dim 2048 \
    --decoder-type mlp \
    --freeze-encoder \
    --unfreeze-epoch 6 \
    --use-adapter \
    --output-dir results/ablation_peft_unfreeze \
    --early-stopping-patience 10

In [ ]:
# Cấu hình 3: + Refine Decoder (Giải mã 2 tầng Coarse-to-Fine)
!python src/training/training_pipeline.py \
    --dataset-mode processed \
    --processed-dir data/processed_chair \
    --split train \
    --val-split val \
    --categories chair \
    --num-points 2048 \
    --epochs 80 \
    --batch-size 32 \
    --encoder-name resnet50 \
    --feature-dim 2048 \
    --decoder-type refine_mlp \
    --freeze-encoder \
    --unfreeze-epoch 6 \
    --use-adapter \
    --output-dir results/ablation_refine \
    --early-stopping-patience 10

In [ ]:
# Cấu hình 4: Full Model (+ Bật Loss Nâng cao Detail Coverage & Uniformity)
!python src/training/training_pipeline.py \
    --dataset-mode processed \
    --processed-dir data/processed_chair \
    --split train \
    --val-split val \
    --categories chair \
    --num-points 2048 \
    --epochs 80 \
    --batch-size 32 \
    --encoder-name resnet50 \
    --feature-dim 2048 \
    --decoder-type refine_mlp \
    --freeze-encoder \
    --unfreeze-epoch 6 \
    --use-adapter \
    --detail-coverage-weight 0.05 \
    --uniformity-weight 0.02 \
    --output-dir results/ablation_full \
    --early-stopping-patience 15

## 4. Tổng hợp Metric & Trực quan hóa (Visual Comparison)
Tạo bảng so sánh và xuất hình ảnh 3 mẫu từ tập dữ liệu.

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import torch
import numpy as np
from PIL import Image
import glob

# Tổng hợp Metrics
runs = [
    ('Baseline', 'results/ablation_baseline/outputs/baseline_summary.json'),
    ('+ PEFT + Unfreeze', 'results/ablation_peft_unfreeze/outputs/baseline_summary.json'),
    ('+ Refine Decoder', 'results/ablation_refine/outputs/baseline_summary.json'),
    ('Full (All Losses)', 'results/ablation_full/outputs/baseline_summary.json')
]

metrics_data = []
for name, path in runs:
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            metrics_data.append({
                'Configuration': name,
                'Chamfer Distance': data.get('best_score'),
                'Best Epoch': data.get('best_epoch')
            })
    else:
        metrics_data.append({'Configuration': name, 'Chamfer Distance': None, 'Best Epoch': None})

df = pd.DataFrame(metrics_data)
print("=== BẢNG KẾT QUẢ ABLATION STUDY ===")
print(df)

# Chuẩn bị thư mục comparation
comp_dir = 'results/comparation'
os.makedirs(comp_dir, exist_ok=True)

print("\nĐang tạo ảnh Visual Comparison cho Full Model...")
# Lấy bừa 3 ảnh test
test_images = glob.glob('data/processed_chair/images/*.png')[:3]
full_model_ckpt = 'results/ablation_full/outputs/checkpoints/best_model.pt'

if os.path.exists(full_model_ckpt) and len(test_images) >= 3:
    for i, img_path in enumerate(test_images):
        base_name = os.path.basename(img_path).replace('.png', '')
        gt_path = os.path.join('data/processed_chair/points', base_name + '.npy')
        
        out_prefix = os.path.join(comp_dir, f'sample_{i}')
        os.system(f'python src/inference/baseline_inference.py --image {img_path} --checkpoint {full_model_ckpt} --output-dir {comp_dir} --name sample_{i}')
        
        pred_img_path = out_prefix + '.png'
        if os.path.exists(gt_path) and os.path.exists(pred_img_path):
            # Render GT
            from src.utils.visualization import plot_point_cloud
            gt_pts = np.load(gt_path)
            plot_point_cloud(gt_pts, out_prefix + '_gt.png', title='Ground Truth')
            
            # Tạo subplot 1x3
            fig, axs = plt.subplots(1, 3, figsize=(15, 5))
            axs[0].imshow(Image.open(img_path))
            axs[0].set_title('Input Image')
            axs[0].axis('off')
            
            axs[1].imshow(Image.open(out_prefix + '_gt.png'))
            axs[1].set_title('Ground Truth')
            axs[1].axis('off')
            
            axs[2].imshow(Image.open(pred_img_path))
            axs[2].set_title('Predict (Full Model)')
            axs[2].axis('off')
            
            plt.tight_layout()
            plt.savefig(out_prefix + '_comparison.png', dpi=150)
            plt.show()
            print(f"Đã lưu: {out_prefix}_comparison.png")
else:
    print("Chưa train xong Full Model hoặc thiếu ảnh để so sánh.")